## <font color="royalblue">**Procesamiento de lenguaje natural (NLP)**</font>
### <font color="royalblue">**Extracción de Requisitos**</font>

Extracción a través de NLP de requisitos para las ofertas de empleo que se encuentra en el texto general de la descripción.

In [ ]:
import pandas as pd
import spacy
from langdetect import detect
from spacy.matcher import Matcher
from spacy.matcher import PhraseMatcher


In [2]:
# Cargar df de los portales de empleo
df_Adzuna = pd.read_csv("df_Adzuna_skills_ofertas.csv")
df_Indeed = pd.read_csv("df_Indeed_skills_ofertas.csv")
df_LinkedIn = pd.read_csv("df_LinkedIn_skills_ofertas.csv")

In [ ]:
# Crear diccionario de requisitos
DICCIONARIO_REQUISITOS = {
    "modalidad" : {
        "hibrido": ["trabajo híbrido", "modalidad híbrida", "hybrid", "hybrid working", "hybrid model", "híbrid",  "modalitat híbrida", "hybride"],
        "remoto": ["teletrabajo", "trabajo remoto", "remoto", "remote", "remote work", "work from home", "fully remote", "remote-first", "remot", "teletreball", "home office", "télétravail", "travail à distance"],
        "presencial": ["onsite", "on-site", "presencial", "on site", "sur site"]
    },
    "contrato" : {
        "indefinido": ["indefinido", "permanent", "permanente", "indefinit"],
        "temporal": ["temporal", "temporary", "eventual", "temporaire"],
        "prácticas": ["prácticas", "internship", "becario", "trainee", "pràctiques", "pratiques", "savant"],
        "formación": ["formación", "apprenticeship", "formació", "entraînement"],
        "freelance": ["freelance", "autónomo", "self-employed", "contractor", "autònom", "free-lance"],
        "contrato": ["contract", "contrato", "contracte", "contracter"],  # genérico
    },
    "jornada" : {
        "completa": ["jornada completa", "tiempo completo", "dedicación exclusiva", "horario completo", "full-time", "full time", "à temps plein"],
        "parcial": ["tiempo parcial", "media jornada", "medio tiempo", "trabajo reducido", "part-time", "part time", "jornada parcial", "temps partiel"],
    },
    "nivel experiencia" : {
        "becario": ["becario", "prácticas", "intern", "trainee", "savant"],
        "junior": ["junior", "jr"],
        "middle": ["middle", "mid", "semi senior", "ssr", "moyen senior"],
        "senior": ["senior", "sr", "lead", "principal"],
    },
    "nivel educativo" : {
        "doctorado": ["doctorado", "phd", "doctoral", "doctorat"],
        "master": ["máster", "master", "msc", "postgrado", "posgrado", "màster", "maîtrise"],
        "grado": ["grado", "licenciatura", "bachelor", "bsc", "degree", "ingeniería", "universitario", "grau", "llicenciatura", "degré", "ingénierie", ],
        "fp": ["fp", "formación profesional", "ciclo formativo", "vocational", "formation professionnelle"],
        "bootcamp": ["bootcamp", "curso intensivo", "certificación", "attestation", "microcredenciales", "certificado", "digital credentials", "micro-credentials", "microcredentials",
                "credenciales digitales", "mcro-certifications", "microcertificaciones", "short learning programmes", "Badges", "microcredencials"]
    }
} 


In [6]:
# Cargar los modelos de lenguaje para español, inglés y catalan
nlp_es = spacy.load("es_core_news_md")
nlp_en = spacy.load("en_core_web_md")
nlp_ca = spacy.load("ca_core_news_md")
nlp_fr = spacy.load("fr_core_news_md")

modelos = { "es": nlp_es,
            "ca": nlp_ca,
            "en": nlp_en,
            "fr": nlp_fr}

# Función para detectar el idioma de un texto
def detectar_idioma(texto):
    try:
        return detect(texto)
    except:
        return "unknown"

In [ ]:
# Funcion para crear matchers por idioma
def crear_matchers_por_idioma(modelos, requisitos_dict):
    matchers_por_idioma = {}

    for idioma, nlp in modelos.items():
        matchers_por_idioma[idioma] = {}
        for categoria, subdict in requisitos_dict.items():
            matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
            for etiqueta, patrones in subdict.items():
                docs = [nlp.make_doc(p) for p in patrones]
                matcher.add(etiqueta.upper(), docs)
            matchers_por_idioma[idioma][categoria] = matcher

    return matchers_por_idioma

matchers_por_idioma = crear_matchers_por_idioma(modelos, DICCIONARIO_REQUISITOS)


In [ ]:
# Función para extraer un requisito de un texto
def extraer_requisito(texto, categoria):
    if pd.isna(texto):
        return None
    
    idioma = detectar_idioma(texto)
    nlp = modelos[idioma]
    matcher = matchers_por_idioma[idioma][categoria]

    doc = nlp(texto)
    matches = matcher(doc)

    if matches:
        match_id, start, end = matches[0]
        return nlp.vocab.strings[match_id].lower()
    
    return None
# usa detectar_idioma

In [9]:
# Patrones para la extracción de la experiencia
pattern_exp = [
    {"LIKE_NUM": True},
    {"LOWER": {"IN": ["años", "año", "anys", "any", "years", "year", "année", "années"]}}
]

# Creacion de matcher para cada idioma
matcher_exp = {
    "es": Matcher(nlp_es.vocab),
    "ca": Matcher(nlp_ca.vocab),
    "en": Matcher(nlp_en.vocab),
    "fr": Matcher(nlp_fr.vocab)
}

for idioma in matcher_exp:
    matcher_exp[idioma].add("EXPERIENCIA", [pattern_exp])


In [ ]:
# Función para extraer la experiencia en años
def extraer_experiencia(texto):
    if pd.isna(texto):
        return None
    
    idioma = detectar_idioma(texto)
    nlp = modelos[idioma]
    doc = nlp(texto)
    matches = matcher_exp[idioma](doc)

    if matches:
        _, start, end = matches[0]
        return doc[start].text  # número de años
    
    return None
# usa detectar_idioma

ADZUNA

In [ ]:
# Extracción de requisitos_nlp para Adzuna
df_Adzuna["modalidad_nlp"] = df_Adzuna["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "modalidad"))
df_Adzuna["contrato_nlp"]  = df_Adzuna["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "contrato"))
df_Adzuna["educacion_nlp"] = df_Adzuna["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "nivel educativo"))
df_Adzuna["experiencia_nivel_nlp"] = df_Adzuna["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "nivel experiencia"))
df_Adzuna["jornada_nlp"] = df_Adzuna["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "jornada"))
df_Adzuna["experiencia_años_nlp"] = df_Adzuna["descripcion_nlp"].apply(lambda x: extraer_experiencia(x))    


INDEED

In [ ]:
# Extracción de requisitos_nlp para Indeed
df_Indeed["modalidad_nlp"] = df_Indeed["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "modalidad"))
df_Indeed["contrato_nlp"]  = df_Indeed["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "contrato"))
df_Indeed["educacion_nlp"] = df_Indeed["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "nivel educativo"))
df_Indeed["experiencia_nivel_nlp"] = df_Indeed["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "nivel experiencia"))
df_Indeed["jornada_nlp"] = df_Indeed["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "jornada"))   
df_Indeed["experiencia_años_nlp"] = df_Indeed["descripcion_nlp"].apply(extraer_experiencia)

LINKEDIN

In [ ]:
# Extracción de requisitos_nlp para LinkedIn
df_LinkedIn["modalidad_nlp"] = df_LinkedIn["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "modalidad"))
df_LinkedIn["contrato_nlp"]  = df_LinkedIn["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "contrato"))
df_LinkedIn["educacion_nlp"] = df_LinkedIn["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "nivel educativo"))
df_LinkedIn["experiencia_nivel_nlp"] = df_LinkedIn["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "nivel experiencia"))
df_LinkedIn["jornada_nlp"] = df_LinkedIn["descripcion_nlp"].apply(lambda x: extraer_requisito(x, "jornada"))
df_LinkedIn["experiencia_años_nlp"] = df_LinkedIn["descripcion_nlp"].apply(extraer_experiencia)

In [ ]:
# Exportar dataframes para estructurar datos finales
df_Adzuna.to_csv("df_Adzuna_ofertas.csv", index=False)
df_Indeed.to_csv("df_Indeed_ofertas.csv", index=False)
df_LinkedIn.to_csv("df_LinkedIn_ofertas.csv", index=False)